# ninja smoke test

In [ ]:
# Example 1: Check ninja version
import ninja
print(ninja.__version__)


In [ ]:
# Example 2: Show ninja executable path
import ninja
print(ninja.BIN_DIR)


In [ ]:
# Example 3: Run ninja from Python (subprocess)
import subprocess
import ninja
try:
    result = subprocess.run([ninja.BIN_DIR + '/ninja', '--version'], capture_output=True, text=True)
    print(result.stdout)
except Exception as e:
    print('Ninja not found or not executable:', e)
    raise AssertionError(
        f"ninja example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

In [ ]:
# Example 4: Show ninja module doc
import ninja
print(ninja.__doc__)


In [ ]:
# Example 5: Build a one-rule build.ninja in a temporary directory
import os
import subprocess
import tempfile
import ninja

ninja_exe = os.path.join(ninja.BIN_DIR, 'ninja')

with tempfile.TemporaryDirectory() as build_dir:
    with open(os.path.join(build_dir, 'in.txt'), 'w', encoding='utf-8') as f:
        f.write('hello from ninja\n')
    with open(os.path.join(build_dir, 'build.ninja'), 'w', encoding='utf-8') as f:
        f.write(
            'rule copy\n'
            '  command = cp $in $out\n'
            '  description = COPY $out\n'
            'build out.txt: copy in.txt\n'
            'default out.txt\n'
        )

    targets = subprocess.run([ninja_exe, '-C', build_dir, '-t', 'targets', 'all'],
                             capture_output=True, text=True, check=True)
    print(targets.stdout.strip())
    assert 'out.txt: copy' in targets.stdout, targets.stdout

    first = subprocess.run([ninja_exe, '-C', build_dir], capture_output=True, text=True, check=True)
    print(first.stdout.strip())
    with open(os.path.join(build_dir, 'out.txt'), encoding='utf-8') as f:
        assert f.read() == 'hello from ninja\n'

    # Nothing changed, so a second run has no work to do
    second = subprocess.run([ninja_exe, '-C', build_dir], capture_output=True, text=True, check=True)
    print(second.stdout.strip())
    assert 'no work to do' in second.stdout, second.stdout